# Clase 8 — Estadística aplicada con Python

**Python y Políticas Públicas**

---

## Contenidos
1. Estadística descriptiva con pandas y scipy
2. Correlaciones e interpretación
3. Regresión lineal simple con statsmodels (OLS)
4. Regresión lineal múltiple
5. Tablas de resultados para un policy brief
6. Ejercicios

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import scipy.stats as stats
import statsmodels.api as sm
import statsmodels.formula.api as smf

%matplotlib inline
pd.set_option('display.float_format', '{:.4f}'.format)

# Dataset provincial (mismo que clases anteriores, expandido)
np.random.seed(42)
n = 24
provincias = [
    "CABA", "Buenos Aires", "Córdoba", "Santa Fe", "Mendoza",
    "Tucumán", "Salta", "Entre Ríos", "Chaco", "Misiones",
    "Corrientes", "Santiago del Estero", "Jujuy", "Río Negro", "Neuquén",
    "Formosa", "San Juan", "San Luis", "Catamarca", "La Rioja",
    "La Pampa", "Chubut", "Santa Cruz", "Tierra del Fuego"
]

gasto_educ = np.random.uniform(12, 28, n)
gasto_salud = np.random.uniform(8, 20, n)

df = pd.DataFrame({
    'provincia': provincias,
    'pobreza': np.random.uniform(15, 58, n).round(1),
    'desempleo': np.random.uniform(3, 15, n).round(1),
    'pbi_pc': np.random.uniform(4, 18, n).round(1),
    'gasto_educacion': gasto_educ.round(1),
    'gasto_salud': gasto_salud.round(1),
    'anios_escolaridad': np.random.uniform(7, 13, n).round(1),
    'mortalidad_infantil': np.random.uniform(5, 25, n).round(1),
    'cobertura_agua': np.random.uniform(65, 99, n).round(1),
})

# Introducir correlaciones realistas
df['pobreza'] = (65 - df['pbi_pc'] * 2.5 - df['anios_escolaridad'] * 1.5
                 + np.random.normal(0, 4, n)).clip(10, 65).round(1)
df['mortalidad_infantil'] = (30 - df['cobertura_agua'] * 0.2 + df['pobreza'] * 0.2
                              + np.random.normal(0, 2, n)).clip(3, 30).round(1)

print(f"Dataset: {df.shape}")
df.head()

---
## 1. Estadística descriptiva

Antes de cualquier modelo, siempre hay que conocer bien los datos.

In [ ]:
# Estadísticas descriptivas completas
desc = df.select_dtypes(include='number').describe().T
desc['cv'] = (desc['std'] / desc['mean'] * 100).round(1)  # coeficiente de variación
desc.round(2)

In [ ]:
# Percentiles adicionales
for var in ['pobreza', 'pbi_pc', 'mortalidad_infantil']:
    q = df[var].quantile([0.1, 0.25, 0.5, 0.75, 0.9])
    print(f"{var}: p10={q[0.1]:.1f}, p25={q[0.25]:.1f}, mediana={q[0.5]:.1f}, p75={q[0.75]:.1f}, p90={q[0.9]:.1f}")

In [ ]:
# Tests estadísticos básicos con scipy

# Test de normalidad (Shapiro-Wilk)
for var in ['pobreza', 'pbi_pc']:
    stat, p = stats.shapiro(df[var])
    interpretacion = "no rechaza normalidad" if p > 0.05 else "rechaza normalidad"
    print(f"Shapiro-Wilk ({var}): W={stat:.3f}, p={p:.3f} → {interpretacion}")

---
## 2. Correlaciones e interpretación

In [ ]:
# Matriz de correlación de Pearson
corr_vars = ['pobreza', 'desempleo', 'pbi_pc', 'gasto_educacion', 'anios_escolaridad', 'mortalidad_infantil']
corr = df[corr_vars].corr()
corr.round(3)

In [ ]:
# Correlación con test de significancia
print(f"{'Variable':<25} {'r':>6} {'p-valor':>10} {'Significativa':>15}")
print("-" * 60)

for var in corr_vars:
    if var == 'pobreza':
        continue
    r, p = stats.pearsonr(df['pobreza'].dropna(), df[var].dropna())
    sig = "✓ (p<0.05)" if p < 0.05 else "✗"
    print(f"{var:<25} {r:>6.3f} {p:>10.3f} {sig:>15}")

In [ ]:
# Heatmap de correlaciones
fig, ax = plt.subplots(figsize=(8, 7))

im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax)

etiquetas = ['Pobreza', 'Desempleo', 'PBI pc', 'Gasto educ.', 'Escolaridad', 'Mort. infantil']
ax.set_xticks(range(len(corr_vars)))
ax.set_yticks(range(len(corr_vars)))
ax.set_xticklabels(etiquetas, rotation=45, ha='right')
ax.set_yticklabels(etiquetas)

for i in range(len(corr_vars)):
    for j in range(len(corr_vars)):
        color = 'white' if abs(corr.values[i, j]) > 0.5 else 'black'
        ax.text(j, i, f"{corr.values[i, j]:.2f}", ha='center', va='center', fontsize=9, color=color)

ax.set_title("Correlaciones entre indicadores socioeconómicos provinciales", fontsize=12, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

---
## 3. Regresión lineal simple (OLS)

La regresión OLS (Mínimos Cuadrados Ordinarios) estima la relación lineal entre una variable dependiente (Y) y una o más variables independientes (X).

**Interpretación**: el coeficiente β₁ indica cuánto cambia Y en promedio cuando X aumenta en 1 unidad, manteniendo lo demás constante.

In [ ]:
# ¿Cómo se relaciona el PBI per cápita con la tasa de pobreza?
modelo_simple = smf.ols('pobreza ~ pbi_pc', data=df).fit()
print(modelo_simple.summary())

In [ ]:
# Extraer los resultados clave
print("=" * 50)
print("RESULTADOS PRINCIPALES")
print("=" * 50)
print(f"Intercepto (α):   {modelo_simple.params['Intercept']:.2f}")
print(f"Coeficiente PBI:  {modelo_simple.params['pbi_pc']:.2f}")
print(f"  → Por cada 1.000$ más de PBI per cápita, la tasa de pobreza")
print(f"    varía en {modelo_simple.params['pbi_pc']:.2f} puntos porcentuales")
print(f"p-valor (PBI):    {modelo_simple.pvalues['pbi_pc']:.4f}")
print(f"R² ajustado:      {modelo_simple.rsquared_adj:.3f}")
print(f"  → El modelo explica el {modelo_simple.rsquared_adj*100:.1f}% de la varianza en pobreza")

In [ ]:
# Visualizar la regresión
fig, ax = plt.subplots(figsize=(9, 6))

ax.scatter(df['pbi_pc'], df['pobreza'], alpha=0.8, color='steelblue', s=80, zorder=3)

# Línea de regresión
x_range = np.linspace(df['pbi_pc'].min(), df['pbi_pc'].max(), 100)
y_pred = modelo_simple.params['Intercept'] + modelo_simple.params['pbi_pc'] * x_range
ax.plot(x_range, y_pred, color='red', linewidth=2, label=f'OLS: pobreza = {modelo_simple.params["Intercept"]:.1f} + {modelo_simple.params["pbi_pc"]:.2f}·PBI')

# Etiquetar outliers (residuos grandes)
df['residuo'] = modelo_simple.resid
outliers = df[df['residuo'].abs() > df['residuo'].abs().quantile(0.85)]
for _, row in outliers.iterrows():
    ax.annotate(row['provincia'], (row['pbi_pc'], row['pobreza']),
                textcoords='offset points', xytext=(6, 4), fontsize=8, color='gray')

ax.set_title(f"PBI per cápita vs. pobreza provincial (R²={modelo_simple.rsquared:.2f})",
             fontsize=13, fontweight='bold')
ax.set_xlabel("PBI per cápita (miles de pesos)")
ax.set_ylabel("Tasa de pobreza (%)")
ax.legend()
ax.grid(alpha=0.3)
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)

plt.tight_layout()
plt.show()

---
## 4. Regresión lineal múltiple

In [ ]:
# Modelo con múltiples predictores
modelo_multiple = smf.ols(
    'pobreza ~ pbi_pc + anios_escolaridad + gasto_educacion + desempleo',
    data=df
).fit()

print(modelo_multiple.summary())

In [ ]:
# Comparar modelos
print(f"{'Modelo':<40} {'R² ajustado':>12} {'AIC':>10}")
print("-" * 65)
print(f"{'Simple (solo PBI pc)':<40} {modelo_simple.rsquared_adj:>12.3f} {modelo_simple.aic:>10.1f}")
print(f"{'Múltiple (PBI + educ + desempleo)':<40} {modelo_multiple.rsquared_adj:>12.3f} {modelo_multiple.aic:>10.1f}")

---
## 5. Tablas de resultados para un policy brief

En un policy brief necesitamos presentar los resultados de forma clara y concisa, sin toda la salida técnica de `summary()`.

In [ ]:
def tabla_regresion(modelo, nombre_vars=None):
    """Genera una tabla de regresión limpia para un policy brief."""
    params = modelo.params
    pvalues = modelo.pvalues
    conf = modelo.conf_int()
    
    rows = []
    for var in params.index:
        p = pvalues[var]
        sig = '***' if p < 0.01 else ('**' if p < 0.05 else ('*' if p < 0.1 else ''))
        rows.append({
            'Variable': nombre_vars.get(var, var) if nombre_vars else var,
            'Coeficiente': f"{params[var]:.3f}{sig}",
            'IC 95%': f"[{conf.loc[var, 0]:.3f}, {conf.loc[var, 1]:.3f}]",
            'p-valor': f"{p:.3f}",
        })
    
    tabla = pd.DataFrame(rows)
    print(tabla.to_string(index=False))
    print(f"\nObservaciones: {int(modelo.nobs)}")
    print(f"R² ajustado: {modelo.rsquared_adj:.3f}")
    print("Significancia: *** p<0.01, ** p<0.05, * p<0.1")
    return tabla

nombres = {
    'Intercept': 'Constante',
    'pbi_pc': 'PBI per cápita (miles $)',
    'anios_escolaridad': 'Años de escolaridad promedio',
    'gasto_educacion': 'Gasto en educación (% presupuesto)',
    'desempleo': 'Tasa de desempleo (%)',
}

print("Tabla 1. Determinantes de la tasa de pobreza provincial")
print("Variable dependiente: Tasa de pobreza (%)")
print("="*65)
tabla_regresion(modelo_multiple, nombres)

---
## 6. Ejercicios

### Ejercicio 1
Estimá un modelo OLS simple para explicar la `mortalidad_infantil` en función de la `cobertura_agua`. Interpretá el coeficiente y calculá el R².

In [ ]:
# Tu solución aquí


### Ejercicio 2
Construí un modelo múltiple para explicar la `mortalidad_infantil` usando al menos 3 predictores. Comparalo con el modelo simple del ejercicio anterior usando R² ajustado.

In [ ]:
# Tu solución aquí


### Ejercicio 3
Usando la función `tabla_regresion()` definida en esta clase, generá una tabla de resultados lista para incluir en un policy brief para el modelo múltiple del ejercicio 2.

In [ ]:
# Tu solución aquí
